In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import classification_report
import os
import numpy as np

In [10]:
data_dir = "../data/card_dataset"
batch_size = 32
image_size = (224, 224)

In [11]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(data_dir, 'train'),
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory(
    os.path.join(data_dir, 'valid'),
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(data_dir, 'test'),
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 7624 images belonging to 53 classes.
Found 265 images belonging to 53 classes.
Found 265 images belonging to 53 classes.


In [12]:
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Clases: {class_names}")

Clases: ['ace of clubs', 'ace of diamonds', 'ace of hearts', 'ace of spades', 'eight of clubs', 'eight of diamonds', 'eight of hearts', 'eight of spades', 'five of clubs', 'five of diamonds', 'five of hearts', 'five of spades', 'four of clubs', 'four of diamonds', 'four of hearts', 'four of spades', 'jack of clubs', 'jack of diamonds', 'jack of hearts', 'jack of spades', 'joker', 'king of clubs', 'king of diamonds', 'king of hearts', 'king of spades', 'nine of clubs', 'nine of diamonds', 'nine of hearts', 'nine of spades', 'queen of clubs', 'queen of diamonds', 'queen of hearts', 'queen of spades', 'seven of clubs', 'seven of diamonds', 'seven of hearts', 'seven of spades', 'six of clubs', 'six of diamonds', 'six of hearts', 'six of spades', 'ten of clubs', 'ten of diamonds', 'ten of hearts', 'ten of spades', 'three of clubs', 'three of diamonds', 'three of hearts', 'three of spades', 'two of clubs', 'two of diamonds', 'two of hearts', 'two of spades']


In [13]:
# Cargar ResNet50 sin la capa final
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

In [14]:
# Modelo final
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("=== Entrenamiento inicial ===")
model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=5
)

=== Entrenamiento inicial ===
Epoch 1/5


C:\Users\pguer\Desktop\Personales\Universidad\2025-I\Movil\stress-predict\backend\ml-service\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


239/239 ━━━━━━━━━━━━━━━━━━━━ 163s 665ms/step - accuracy: 0.1917 - loss: 3.2311 - val_accuracy: 0.4679 - val_loss: 1.8547
Epoch 2/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 146s 609ms/step - accuracy: 0.5093 - loss: 1.6429 - val_accuracy: 0.4943 - val_loss: 1.6263
Epoch 3/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 138s 578ms/step - accuracy: 0.6625 - loss: 1.1703 - val_accuracy: 0.5509 - val_loss: 1.5524
Epoch 4/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 146s 609ms/step - accuracy: 0.7372 - loss: 0.9335 - val_accuracy: 0.5321 - val_loss: 1.4721
Epoch 5/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 144s 602ms/step - accuracy: 0.7925 - loss: 0.7583 - val_accuracy: 0.5736 - val_loss: 1.5327


In [15]:
# Fine tuning
for layer in base_model.layers:
    if 'conv5' in layer.name or 'bn5' in layer.name:  # solo las últimas capas
        layer.trainable = True

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

In [16]:
print("=== Fine-tuning ===")
model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=3
)

=== Fine-tuning ===
Epoch 1/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 215s 873ms/step - accuracy: 0.6934 - loss: 1.0395 - val_accuracy: 0.7547 - val_loss: 0.7328
Epoch 2/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 199s 833ms/step - accuracy: 0.9711 - loss: 0.1416 - val_accuracy: 0.8226 - val_loss: 0.5943
Epoch 3/3
239/239 ━━━━━━━━━━━━━━━━━━━━ 199s 834ms/step - accuracy: 0.9909 - loss: 0.0529 - val_accuracy: 0.8377 - val_loss: 0.5887


In [17]:
test_generator.reset()
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes

print("\nReporte de clasificación final en Test:")
print(classification_report(y_true, y_pred, target_names=class_names))

9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 681ms/step

Reporte de clasificación final en Test:
                   precision    recall  f1-score   support

     ace of clubs       0.83      1.00      0.91         5
  ace of diamonds       1.00      1.00      1.00         5
    ace of hearts       0.83      1.00      0.91         5
    ace of spades       1.00      1.00      1.00         5
   eight of clubs       0.71      1.00      0.83         5
eight of diamonds       0.50      1.00      0.67         5
  eight of hearts       0.67      0.80      0.73         5
  eight of spades       1.00      1.00      1.00         5
    five of clubs       1.00      0.40      0.57         5
 five of diamonds       1.00      0.60      0.75         5
   five of hearts       0.83      1.00      0.91         5
   five of spades       1.00      0.80      0.89         5
    four of clubs       0.62      1.00      0.77         5
 four of diamonds       0.83      1.00      0.91         5
   four of hearts       1.00      